# Train WATERec in Jupyter

Run top to bottom with the bundled plate example or set `WATEREC_CONFIG` to choose another profile. To try the public energy-meter data, run [the download notebook](00_download_ocr_example.ipynb) first and select `configs/energy_meter.yaml`.

The cells expose the dataset, model, loss, optimizer and predictions. Edit `prepare_image` to try augmentation, call `train_step` on one batch to debug, or change `infer` and compare predictions before and after training. The model backbone and checkpoint format come from the WATERec Python package.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import random
import sys
from contextlib import nullcontext

import numpy as np
from PIL import Image, ImageEnhance
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# An installed waterec_ocr package also works when running outside the checkout.
if importlib.util.find_spec("waterec_ocr") is None:
    location = Path.cwd().resolve()
    checkout = next((p for p in (location, *location.parents)
                     if (p / "waterec_ocr").is_dir()), None)
    if checkout is None:
        raise ImportError("Install waterec_ocr or open the notebook from the checkout")
    sys.path.insert(0, str(checkout))

from waterec_ocr.data_io import scan_pairs
from waterec_ocr.data import audit_splits
from waterec_ocr.metrics import EXACT_POLICY
from waterec_ocr.waterec_charset import metadata as alphabet_metadata
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import create_native, preprocess_image
from waterec_ocr.waterec_synthetic import generate_replay
from waterec_ocr.waterec_training import read_manifest, split_training_records

## Choose the dataset and training settings

In [ ]:
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
INITIAL_CHECKPOINT = None  # Optional compatible fine-tuned .pt file, with a fresh optimizer.

# Edit or uncomment settings for this run.
# config["dataset_root"] = Path("/path/to/dataset")  # train/ and val/ contain image/TXT pairs
# config["output_dir"] = Path("artifacts/my_experiment")
# config["epochs"] = 5
# config["warmup_epochs"] = 2  # Set to epochs to train the decoder only.
# config["device"], config["precision"] = "cuda", "bf16"
# config["batch_size"] = 4

if config["normalization"] != "none":
    raise ValueError("Use literal labels with normalization set to none.")
device = torch.device(config["device"])
if config["precision"] == "bf16" and (device.type != "cuda" or not torch.cuda.is_bf16_supported()):
    raise ValueError("Select fp32 or a CUDA device with BF16 support.")
DATASET = Path(config["dataset_root"])
OUTPUT = Path(config["output_dir"])
print(config["variant"], f'{config["tokens"]} patches', device)
print("Dataset:", DATASET, "Output:", OUTPUT)

## Inspect the data

The epoch is selected using a holdout from the training pool. The separate `val/` directory is opened after selection.

In [ ]:
records = (read_manifest(config["train_manifest"]) if config.get("train_manifest")
           else scan_pairs(DATASET / "train", "none"))
real = [row for row in records if not row.get("synthetic")]
replay = [row for row in records if row.get("synthetic")]
if config.get("split_manifest"):
    split = json.loads(Path(config["split_manifest"]).read_text())
    fit_ids, held_ids = set(split["train"]), set(split["holdout"])
    if fit_ids & held_ids:
        raise ValueError("Fit and holdout IDs overlap.")
    fit = [row for row in real if row["image"] in fit_ids or Path(row["image"]).name in fit_ids]
    holdout = [row for row in real if row["image"] in held_ids or Path(row["image"]).name in held_ids]
    if len(fit) + len(holdout) != len(real) or not fit or not holdout:
        raise ValueError("Split must partition all real training images.")
else:
    fit, holdout = split_training_records(real, config)
if config.get("synthetic_manifest"):
    replay += read_manifest(config["synthetic_manifest"])
if config.get("synthetic_plate_count", 0) and not replay:
    if config["task"] != "plate":
        raise ValueError("Automatic synthetic replay is for plates.")
    replay, _ = generate_replay(OUTPUT / "synthetic_replay", config["synthetic_plate_count"],
                                config.get("synthetic_seed", 20261114),
                                config.get("synthetic_font_dir"))
if len({row["image"] for row in real + replay}) != len(real) + len(replay):
    raise ValueError("Each training image ID must be unique.")
if audit_splits(fit + replay, holdout)["pixel_duplicates_across_splits"]:
    raise ValueError("Fit and holdout have identical decoded images.")
print(f"{len(fit)} training, {len(holdout)} holdout, {len(replay)} synthetic")
example = holdout[0]
with Image.open(example["path"]) as image:
    display(image.copy())
print(repr(example["label"]))

## Prepare batches and load the model

In [ ]:
def prepare_image(image, augment=False):
    if augment:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
    return preprocess_image(image, config["tokens"])


class OCRDataset(Dataset):
    def __init__(self, rows, augment=False):
        self.rows = rows
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as source:
            image = prepare_image(source.convert("RGB"), self.augment)
        return image, row["label"], row["image"]


def collate_images(samples):
    images, labels, names = zip(*samples)
    return list(images), list(labels), list(names)


def make_loader(rows, training=False):
    return DataLoader(OCRDataset(rows, augment=training), batch_size=config["batch_size"],
                      shuffle=training, num_workers=config.get("num_workers", 0),
                      collate_fn=collate_images, pin_memory=device.type == "cuda")


def new_model():
    seed = config.get("seed", 20261114)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    return create_native(config, checkpoint=INITIAL_CHECKPOINT).to(device)


model = new_model()
for row in real + replay:
    model.encode_labels([row["label"]])  # Raise on unsupported characters or oversized labels.
train_loader = make_loader(fit + replay, training=True)
holdout_loader = make_loader(holdout)
image_tensors, target_strings, image_names = next(iter(holdout_loader))
print([tuple(image.shape) for image in image_tensors], target_strings)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Run a prediction before training

In [ ]:
@torch.inference_mode()
def infer(image):
    model.eval()
    pixels = prepare_image(image.convert("RGB")).to(device)
    return model.decode(model([pixels]))[0]


@torch.inference_mode()
def evaluate(loader):
    model.eval()
    predictions = []
    for images, labels, names in loader:
        texts = model.decode(model([image.to(device) for image in images]))
        predictions.extend({"image": name, "target": target, "prediction": text}
                           for name, target, text in zip(names, labels, texts, strict=True))
    return predictions


with Image.open(example["path"]) as image:
    print("Before training:", repr(infer(image)), "Target:", repr(example["label"]))

## Edit the loss, optimizer and training loop

`train_step` returns shapes for inspection. Calling it directly updates weights. Re-run the model cell to start a fresh experiment.

In [ ]:
def make_optimizer():
    return torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": config["encoder_learning_rate"]},
         {"params": model.decoder.parameters(), "lr": config["decoder_learning_rate"]}],
        weight_decay=config.get("weight_decay", 0.01),
    )


def train_step(batch, optimizer):
    images, labels, _ = batch
    images = [image.to(device, non_blocking=True) for image in images]
    targets = model.encode_labels(labels, device)
    optimizer.zero_grad(set_to_none=True)
    precision = (torch.autocast("cuda", dtype=torch.bfloat16)
                 if config["precision"] == "bf16" else nullcontext())
    with precision:
        memory = model.encoder(images)
        logits = model.decoder.forward_train(memory, targets)
        loss = F.cross_entropy(logits.float().flatten(0, 1), targets[:, 1:].flatten(),
                               ignore_index=model.decoder.ignore_index)
    if not torch.isfinite(loss):
        raise RuntimeError("Nonfinite training loss.")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
    optimizer.step()
    return loss.item(), tuple(logits.shape), tuple(targets.shape)


def train_epoch(loader, optimizer, epoch):
    model.train()
    model.encoder.requires_grad_(epoch > config.get("warmup_epochs", 0))
    losses = []
    for batch in loader:
        loss, logits_shape, targets_shape = train_step(batch, optimizer)
        losses.append(loss)
    return {"loss": float(np.mean(losses)), "logits_shape": logits_shape,
            "targets_shape": targets_shape, "encoder_frozen": epoch <= config.get("warmup_epochs", 0)}


def save_checkpoint(path, epoch):
    keys = ("task", "variant", "tokens", "include_space", "additional_characters",
            "max_label_length", "max_decode_steps", "normalization")
    saved_config = {key: model.config[key] for key in keys if key in model.config}
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "config": saved_config,
                "selection_policy": EXACT_POLICY, "epoch": epoch,
                **alphabet_metadata(model)}, path)


checkpoint = OUTPUT / "selection" / "best.pt"
if config.get("refit_all_training"):
    final_checkpoint = OUTPUT / "full_fit" / "model.pt"
else:
    final_checkpoint = checkpoint
if checkpoint.exists() or final_checkpoint.exists():
    raise FileExistsError("Choose a new output_dir to keep earlier checkpoints.")

optimizer = make_optimizer()
history = []
best_exact = -1
for epoch in range(1, config["epochs"] + 1):
    metrics = train_epoch(train_loader, optimizer, epoch)
    matches = sum(r["target"] == r["prediction"] for r in evaluate(holdout_loader))
    row = {"epoch": epoch, "exact": matches, "count": len(holdout), **metrics}
    history.append(row)
    if matches > best_exact:
        best_exact, best_epoch = matches, epoch
        save_checkpoint(checkpoint, epoch)
    print(row)

(OUTPUT / "selection" / "metrics.json").write_text(json.dumps(history, indent=2))
print("Selected epoch:", best_epoch, "Checkpoint:", checkpoint)

## Load the selected checkpoint and inspect predictions

If enabled in the profile, refit from the original initialization on all training rows for the selected number of epochs. Evaluate the untouched validation directory only after training.

In [ ]:
if config.get("refit_all_training"):
    model = new_model()
    optimizer = make_optimizer()
    full_loader = make_loader(real + replay, training=True)
    for epoch in range(1, best_epoch + 1):
        print("Refit", epoch, train_epoch(full_loader, optimizer, epoch))
    save_checkpoint(final_checkpoint, best_epoch)

model = create_native(config, checkpoint=final_checkpoint).to(device)
with Image.open(example["path"]) as image:
    print("After training:", repr(infer(image)), "Target:", repr(example["label"]))

if (DATASET / "val").is_dir():
    validation = evaluate(make_loader(scan_pairs(DATASET / "val", "none")))
    correct = sum(row["target"] == row["prediction"] for row in validation)
    with (OUTPUT / "validation_predictions.jsonl").open("w", encoding="utf-8") as file:
        for row in validation:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Validation exact match: {correct}/{len(validation)} ({correct / len(validation):.1%})")
    display([row for row in validation if row["target"] != row["prediction"]][:10])
print("Use this checkpoint for ONNX export:", final_checkpoint)